# Qwen3 Embedding 0.6B LoRA Fine-Tuning (1024 dim)

This notebook trains a LoRA adapter on pre-generated curriculum pairs and evaluates on fixed test pairs.

Dataset: data_new/curriculum_training_pairs_complete.json + data_new/fixed_test_pairs.json
Base model: Qwen/Qwen3-Embedding-0.6B


In [4]:
import json
import random
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from scipy.stats import spearmanr, pearsonr

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import SentenceEvaluator

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError("peft is required. Install with `pip install peft`.") from e
\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n


In [5]:
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.info


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if torch.cuda.is_available():
    DEVICE = "cuda"
    log(f"[DEVICE] CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    log("[DEVICE] MPS")
else:
    DEVICE = "cpu"
    log("[DEVICE] CPU")


[DEVICE] CUDA: NVIDIA GeForce RTX 5090


In [6]:
CONFIG = {
    "model_name": "Qwen/Qwen3-Embedding-0.6B",
    "trust_remote_code": True,
    "output_dir": "models/qwen3_embedding_lora",
    "train_pairs_path": "data_new/curriculum_training_pairs_complete.json",
    "test_pairs_path": "data_new/fixed_test_pairs.json",
    "use_curriculum": True,
    "epochs": 12,
    "epochs_per_phase": 4,
    "batch_size": 16,
    "lr": 5e-5,
    "warmup_ratio": 0.1,
    "max_seq_length": 256,
    "min_text_length": 25,
    "eval_split": 0.15,
    "holdout_split": 0.10,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "matryoshka_dims": [1024, 768, 512, 256, 128],
    "seed": 42,
}

set_seed(CONFIG["seed"])
log(f"[CONFIG] Model: {CONFIG['model_name']}")
log(f"[CONFIG] Output: {CONFIG['output_dir']}")
log(f"[CONFIG] Matryoshka dims: {CONFIG['matryoshka_dims']}")


[CONFIG] Model: Qwen/Qwen3-Embedding-0.6B
[CONFIG] Output: models/qwen3_embedding_lora
[CONFIG] Matryoshka dims: [1024, 768, 512, 256, 128]


In [7]:

def load_pairs(path: str):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    texts1 = data["texts1"]
    texts2 = data["texts2"]
    labels = data["labels"]
    phase_indicators = data.get("phase_indicators")
    examples = [
        InputExample(texts=[t1, t2], label=float(y))
        for t1, t2, y in zip(texts1, texts2, labels)
    ]
    return examples, phase_indicators, data.get("metadata", {})


train_examples, phase_indicators, train_meta = load_pairs(CONFIG["train_pairs_path"])
log(f"[DATA] Train pairs: {len(train_examples)}")

if CONFIG["use_curriculum"] and phase_indicators:
    phase1, phase2, phase3 = [], [], []
    for ex, phase in zip(train_examples, phase_indicators):
        if phase == 1:
            phase1.append(ex)
        elif phase == 2:
            phase2.append(ex)
        elif phase == 3:
            phase3.append(ex)
    CURRICULUM_PHASES = {"phase1": phase1, "phase2": phase2, "phase3": phase3}
    log(f"[DATA] Phase sizes: {len(phase1)}, {len(phase2)}, {len(phase3)}")
else:
    CURRICULUM_PHASES = None


eval_examples, _, _ = load_pairs(CONFIG["test_pairs_path"])
log(f"[DATA] Eval pairs: {len(eval_examples)}")


[DATA] Train pairs: 15000
[DATA] Phase sizes: 5000, 5000, 5000
[DATA] Eval pairs: 1000


In [8]:

def init_model_with_lora(config, device):
    log(f"[MODEL] Loading {config['model_name']}")
    model = SentenceTransformer(
        config["model_name"],
        device=device,
        trust_remote_code=config.get("trust_remote_code", False),
    )
    model.max_seq_length = min(config["max_seq_length"], model.max_seq_length)

    base_model = model[0].auto_model
    linear_names = [
        name for name, module in base_model.named_modules() if isinstance(module, nn.Linear)
    ]
    configured = config["lora_target_modules"]
    resolved = [t for t in configured if any(t in n for n in linear_names)]
    if not resolved:
        fallback = ["q_proj", "k_proj", "v_proj", "o_proj", "query", "key", "value", "dense"]
        resolved = [t for t in fallback if any(t in n for n in linear_names)]
    if not resolved:
        raise ValueError("No LoRA target modules matched. Check lora_target_modules.")

    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        target_modules=resolved,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )

    peft_model = get_peft_model(base_model, lora_config)
    peft_model = peft_model.to(device)
    model[0].auto_model = peft_model

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    log(f"[MODEL] LoRA applied. Trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
    return model


model = init_model_with_lora(CONFIG, DEVICE)


[MODEL] Loading Qwen/Qwen3-Embedding-0.6B
Load pretrained SentenceTransformer: Qwen/Qwen3-Embedding-0.6B
1 prompt is loaded, with the key: query
[MODEL] LoRA applied. Trainable: 4,587,520 (0.76%)


In [9]:

class ITSMEvaluator(SentenceEvaluator):
    def __init__(self, examples, batch_size=32, name="eval"):
        self.examples = examples
        self.batch_size = batch_size
        self.name = name

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        metrics = evaluate_pairs(model, self.examples, self.batch_size)
        log(
            f"[EVAL] {self.name} epoch={epoch} spearman={metrics['spearman']:.4f} "
            f"roc_auc={metrics['roc_auc']:.4f} f1={metrics['f1']:.4f}"
        )
        return metrics["spearman"]



def evaluate_pairs(model, examples, batch_size=32):
    texts1 = [ex.texts[0] for ex in examples]
    texts2 = [ex.texts[1] for ex in examples]
    labels = np.array([ex.label for ex in examples])

    emb1 = model.encode(texts1, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    emb2 = model.encode(texts2, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    scores = np.sum(emb1 * emb2, axis=1) / (
        np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1) + 1e-8
    )

    spearman = spearmanr(labels, scores).correlation
    pearson = pearsonr(labels, scores).correlation
    roc_auc = roc_auc_score(labels, scores)\n    fpr, tpr, _ = roc_curve(labels, scores)
    pr_auc = average_precision_score(labels, scores)

    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = int(np.argmax(f1_scores))
    best_threshold = thresholds[best_idx - 1] if best_idx > 0 else 0.5

    preds = (scores >= best_threshold).astype(int)
    f1 = f1_score(labels, preds)
    accuracy = accuracy_score(labels, preds)
    precision_at = precision_score(labels, preds)
    recall_at = recall_score(labels, preds)
    cm = confusion_matrix(labels, preds).tolist()

    return {
        "scores": scores.tolist(),
        "labels": labels.tolist(),
        "precision_curve": precision.tolist(),
        "recall_curve": recall.tolist(),
        "thresholds": thresholds.tolist(),
        "f1_curve": f1_scores.tolist(),
        "roc_fpr": fpr.tolist(),
        "roc_tpr": tpr.tolist(),        "spearman": float(spearman),
        "pearson": float(pearson),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "f1": float(f1),
        "accuracy": float(accuracy),
        "precision": float(precision_at),
        "recall": float(recall_at),
        "best_threshold": float(best_threshold),
        "confusion_matrix": cm,
    }



In [10]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_path = Path(CONFIG["output_dir"]) / f"qwen3_1024_{timestamp}"
save_path.mkdir(parents=True, exist_ok=True)

base_loss = losses.MultipleNegativesRankingLoss(model)
train_loss = losses.MatryoshkaLoss(
    model,
    base_loss,
    matryoshka_dims=CONFIG["matryoshka_dims"],
)

evaluator = ITSMEvaluator(eval_examples, batch_size=CONFIG["batch_size"], name="eval")
use_amp = DEVICE == "cuda"

if CONFIG["use_curriculum"] and CURRICULUM_PHASES:
    for phase_name, phase_examples in sorted(CURRICULUM_PHASES.items()):
        phase_loader = DataLoader(
            phase_examples,
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=0,
            pin_memory=(DEVICE == "cuda"),
        )
        total_steps = len(phase_loader) * CONFIG["epochs_per_phase"]
        warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

        log(f"[TRAIN] {phase_name} examples={len(phase_examples)} epochs={CONFIG['epochs_per_phase']}")
        model.fit(
            train_objectives=[(phase_loader, train_loss)],
            evaluator=evaluator,
            epochs=CONFIG["epochs_per_phase"],
            warmup_steps=warmup_steps,
            optimizer_params={"lr": CONFIG["lr"]},
            scheduler="WarmupCosine",
            output_path=str(save_path),
            evaluation_steps=max(100, len(phase_loader) // 2),
            save_best_model=True,
            show_progress_bar=True,
            use_amp=use_amp,
        )
else:
    train_loader = DataLoader(
        train_examples,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )
    total_steps = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    model.fit(
        train_objectives=[(train_loader, train_loss)],
        evaluator=evaluator,
        epochs=CONFIG["epochs"],
        warmup_steps=warmup_steps,
        optimizer_params={"lr": CONFIG["lr"]},
        scheduler="WarmupCosine",
        output_path=str(save_path),
        evaluation_steps=max(100, len(train_loader) // 2),
        save_best_model=True,
        show_progress_bar=True,
        use_amp=use_amp,
    )

log(f"[TRAIN] Complete. Best model in {save_path}")


[TRAIN] phase1 examples=5000 epochs=4


Step,Training Loss,Validation Loss,Evaluator
156,No log,No log,0.444196
312,No log,No log,0.414647
313,No log,No log,0.413615
468,No log,No log,0.386290
624,13.605800,No log,0.359173
626,13.605800,No log,0.355432
780,13.605800,No log,0.393773
936,13.605800,No log,0.378669
939,13.605800,No log,0.375385
1092,12.855300,No log,0.362762


Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.86it/s]
[EVAL] eval epoch=0.4984025559105431 spearman=0.4442 roc_auc=0.7565 f1=0.7210
Save model to models\qwen3_embedding_lora\qwen3_1024_20251228_0300
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.85it/s]
[EVAL] eval epoch=0.9968051118210862 spearman=0.4146 roc_auc=0.7394 f1=0.7145
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.48it/s]
[EVAL] eval epoch=1.0 spearman=0.4136 roc_auc=0.7388 f1=0.7140
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.39it/s]
[EVAL] eval epoch=1.4952076677316293 spearman=0.3863 roc_auc=0.7230 f1=0.7206
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.53it/s]
[EVAL] eval epoch=1.9936102236421727 spearman=0.3592 roc_auc=0.7074 f1=0.6984
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:09<00:00,  6.77it/s]
[EVAL] eval epoch=2.0 spearman=0.3554 roc_auc=0.7052 f1=0.6975
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

Step,Training Loss,Validation Loss,Evaluator
156,No log,No log,0.269276
312,No log,No log,0.232006
313,No log,No log,0.237804
468,No log,No log,0.134692
624,13.976300,No log,0.160576
626,13.976300,No log,0.159183
780,13.976300,No log,0.128543
936,13.976300,No log,0.181558
939,13.976300,No log,0.178932
1092,13.825000,No log,0.140002


Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:07<00:00,  8.09it/s]
[EVAL] eval epoch=0.4984025559105431 spearman=0.2693 roc_auc=0.6555 f1=0.6740
Save model to models\qwen3_embedding_lora\qwen3_1024_20251228_0300
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:07<00:00,  8.20it/s]
[EVAL] eval epoch=0.9968051118210862 spearman=0.2320 roc_auc=0.6339 f1=0.6712
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.66it/s]
[EVAL] eval epoch=1.0 spearman=0.2378 roc_auc=0.6373 f1=0.6721
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:07<00:00,  8.67it/s]
[EVAL] eval epoch=1.4952076677316293 spearman=0.1347 roc_auc=0.5778 f1=0.6667
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.54it/s]
[EVAL] eval epoch=1.9936102236421727 spearman=0.1606 roc_auc=0.5927 f1=0.6676
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:09<00:00,  6.93it/s]
[EVAL] eval epoch=2.0 spearman=0.1592 roc_auc=0.5919 f1=0.6671
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

Step,Training Loss,Validation Loss,Evaluator
156,No log,No log,0.097882
312,No log,No log,0.215291
313,No log,No log,0.213230
468,No log,No log,0.093438
624,13.865000,No log,0.075933
626,13.865000,No log,0.075026
780,13.865000,No log,0.050732
936,13.865000,No log,0.040253
939,13.865000,No log,0.040974
1092,13.798800,No log,0.008300


Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:09<00:00,  6.76it/s]
[EVAL] eval epoch=0.4984025559105431 spearman=0.0979 roc_auc=0.5565 f1=0.6667
Save model to models\qwen3_embedding_lora\qwen3_1024_20251228_0300
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.32it/s]
[EVAL] eval epoch=0.9968051118210862 spearman=0.2153 roc_auc=0.6243 f1=0.6676
Save model to models\qwen3_embedding_lora\qwen3_1024_20251228_0300
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.79it/s]
[EVAL] eval epoch=1.0 spearman=0.2132 roc_auc=0.6231 f1=0.6676
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.32it/s]
[EVAL] eval epoch=1.4952076677316293 spearman=0.0934 roc_auc=0.5539 f1=0.6667
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:08<00:00,  7.21it/s]
[EVAL] eval epoch=1.9936102236421727 spearman=0.0759 roc_auc=0.5438 f1=0.6667
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:09<00:00,  6.93it/s]
[EVAL] eval epoch=2.0 spearma

In [11]:
best_model = SentenceTransformer(str(save_path), device=DEVICE)
eval_metrics = evaluate_pairs(best_model, eval_examples, batch_size=CONFIG["batch_size"])

log(f"[FINAL] Spearman: {eval_metrics['spearman']:.4f}")
log(f"[FINAL] ROC-AUC: {eval_metrics['roc_auc']:.4f}")
log(f"[FINAL] F1: {eval_metrics['f1']:.4f}")
log(f"[FINAL] Best threshold: {eval_metrics['best_threshold']:.4f}")
log(f"[FINAL] Confusion matrix: {eval_metrics['confusion_matrix']}")


Load pretrained SentenceTransformer: models\qwen3_embedding_lora\qwen3_1024_20251228_0300
1 prompt is loaded, with the key: query
Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 63/63 [00:06<00:00,  9.05it/s]
[FINAL] Spearman: 0.2153
[FINAL] ROC-AUC: 0.6243
[FINAL] F1: 0.6676
[FINAL] Best threshold: 0.9963
[FINAL] Confusion matrix: [[2, 498], [0, 500]]


In [ ]:
# Plots: ROC, PR, confusion matrix, score distribution, F1 vs threshold
scores = np.array(eval_metrics["scores"])
labels = np.array(eval_metrics["labels"])

# ROC curve
plt.figure(figsize=(6, 5))
plt.plot(eval_metrics["roc_fpr"], eval_metrics["roc_tpr"], label=f"ROC AUC={eval_metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# PR curve
plt.figure(figsize=(6, 5))
plt.plot(eval_metrics["recall_curve"], eval_metrics["precision_curve"], label=f"PR AUC={eval_metrics['pr_auc']:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Confusion matrix
cm = np.array(eval_metrics["confusion_matrix"])
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Score distribution
plt.figure(figsize=(6, 5))
plt.hist(scores[labels == 0], bins=50, alpha=0.6, label="Negative")
plt.hist(scores[labels == 1], bins=50, alpha=0.6, label="Positive")
plt.xlabel("Similarity Score")
plt.ylabel("Count")
plt.title("Score Distribution")
plt.legend()
plt.tight_layout()
plt.show()

# F1 vs threshold
thr = np.array(eval_metrics["thresholds"])
f1_curve = np.array(eval_metrics["f1_curve"])
if len(f1_curve) == len(thr) + 1:
    f1_plot = f1_curve[1:]
else:
    f1_plot = f1_curve[:len(thr)]
plt.figure(figsize=(6, 5))
plt.plot(thr, f1_plot)
plt.axvline(eval_metrics["best_threshold"], color="r", linestyle="--", label=f"Best={eval_metrics['best_threshold']:.3f}")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.title("F1 vs Threshold")
plt.legend()
plt.tight_layout()
plt.show()


In [12]:
metadata = {
    "timestamp": datetime.now().isoformat(),
    "model_name": CONFIG["model_name"],
    "output_dir": str(save_path),
    "train_pairs_path": CONFIG["train_pairs_path"],
    "test_pairs_path": CONFIG["test_pairs_path"],
    "embedding_dim": 1024,
    "lora": {
        "r": CONFIG["lora_r"],
        "alpha": CONFIG["lora_alpha"],
        "dropout": CONFIG["lora_dropout"],
        "target_modules": CONFIG["lora_target_modules"],
    },
    "training": {
        "epochs": CONFIG["epochs"],
        "epochs_per_phase": CONFIG["epochs_per_phase"],
        "batch_size": CONFIG["batch_size"],
        "lr": CONFIG["lr"],
        "warmup_ratio": CONFIG["warmup_ratio"],
        "max_seq_length": CONFIG["max_seq_length"],
        "use_curriculum": CONFIG["use_curriculum"],
        "matryoshka_dims": CONFIG["matryoshka_dims"],
    },
    "device": DEVICE,
    "metrics": eval_metrics,
}

metadata_path = Path(save_path) / "training_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

log(f"[OK] Metadata saved to {metadata_path}")


[OK] Metadata saved to models\qwen3_embedding_lora\qwen3_1024_20251228_0300\training_metadata.json
